In [16]:
from tensorflow import keras
from keras.models import Model
from keras import layers
from keras.layers import Dense, Input, BatchNormalization, Activation
from keras.layers import Conv2D, SeparableConv2D, MaxPooling2D, GlobalAveragePooling2D, GlobalMaxPooling2D
#from keras.applications.imagenet_utils import _obtain_input_shape
#from keras.utils.data_utils import get_file
from sklearn.model_selection import KFold
from keras.datasets import mnist
from PIL import Image
# for creating a one hot vector for labels
#from keras.utils import np_utils
from IPython.display import display, Image
#import the models
from keras import Model
#add layers
from keras import layers
#add optimizer
from keras import optimizers
#add loss function 
from keras import losses
import random

import keras
import numpy as np
import pandas as pd 
import os
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import json
from skimage.metrics import structural_similarity as ssim

print("Hello world.")

Hello world.


In [17]:
import numpy as np
import cv2

def apply_perturbations(images, perturbation):
    """
    Apply a single perturbation to normalized images.

    Parameters:
    - images: list or array of preprocessed images (H, W, C), normalized [0,1]
    - perturbation: dict specifying type and parameters
        Examples:
        {'type':'gaussian_blur','kernel_size':5}
        {'type':'noise','mean':0,'std':0.05}
        {'type':'brightness','factor':0.2}
        {'type':'hue_shift','shift':10}

    Returns:
    - perturbed_images: np.array of perturbed images
    """
    perturbed_images = []

    for img in images:
        img_copy = img.copy()
        p_type = perturbation.get('type')

        if p_type == 'gaussian_blur':
            k = perturbation.get('kernel_size', 5)
            if k % 2 == 0:
                k += 1
            img_copy = cv2.GaussianBlur(img_copy, (k, k), 0)

        elif p_type == 'noise':
            mean = perturbation.get('mean', 0)
            std = perturbation.get('std', 0.05)
            noise = np.random.normal(mean, std, img_copy.shape)
            img_copy = np.clip(img_copy + noise, 0, 1)

        elif p_type == 'brightness':
            factor = perturbation.get('factor', 0.2)
            img_copy = np.clip(img_copy + factor, 0, 1)

        elif p_type == 'hue_shift':
            hsv = cv2.cvtColor((img_copy * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
            hsv[..., 0] = (hsv[..., 0] + perturbation.get('shift', 10)) % 180
            img_copy = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB).astype("float32") / 255.0

        perturbed_images.append(img_copy)

    return np.array(perturbed_images, dtype=np.float32)




def create_mask_from_json(json_file, img_size):
    # If file does not exist or is empty, return blank mask
    if not os.path.exists(json_file) or os.path.getsize(json_file) == 0:
        print(f"Warning: JSON file missing or empty: {json_file}")
        return np.zeros((img_size, img_size), dtype=np.uint8)

    # read json
    with open(json_file, "r") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            print(f"Warning: JSON file invalid: {json_file}")
            return np.zeros((img_size, img_size), dtype=np.uint8)

    mask = np.zeros((img_size, img_size), dtype=np.uint8)

    # loop over each shape (polygon)
    for shape in data.get("shapes", []):
        pts = np.array(shape["points"], dtype=np.int32)
        pts = pts.reshape((-1, 1, 2))

        # fill polygon
        cv2.fillPoly(mask, [pts], 255)

    return mask

In [18]:
x = []
label = []
x_validation = []
label_validation = []
x_test = []
image_test= []
label_test = []
mask_test = []
img_size = 299

test_range  = []

In [19]:
image_folder0 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Apple"
csv_file0 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/apple_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Apple"


columns = csv_file0.columns
print(list(columns))

for i in range(0, len(csv_file0)):
    image_name = csv_file0.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file0.loc[i]["used_for"]
    
    image = cv2.imread(image_folder0 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([0])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([0])
    else:
        x.append(image)
        label.append([0])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [20]:
image_folder1 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Berry"
csv_file1 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/berry_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Berry"


columns = csv_file1.columns
print(list(columns))

for i in range(0, len(csv_file1)):
    image_name = csv_file1.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file1.loc[i]["used_for"]
    
    image = cv2.imread(image_folder1 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([1])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([1])
    else:
        x.append(image)
        label.append([1])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [21]:
image_folder2 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Fig"
csv_file2 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/fig_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Fig"


columns = csv_file2.columns
print(list(columns))

for i in range(0, len(csv_file2)):
    image_name = csv_file2.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file2.loc[i]["used_for"]
    
    image = cv2.imread(image_folder2 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([2])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([2])
    else:
        x.append(image)
        label.append([2])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [22]:
image_folder3 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Guava"
csv_file3 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/guava_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Guava"


columns = csv_file3.columns
print(list(columns))

for i in range(0, len(csv_file3)):
    image_name = csv_file3.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file3.loc[i]["used_for"]
    
    image = cv2.imread(image_folder3 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([3])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([3])
    else:
        x.append(image)
        label.append([3])

test_range.append(len(x_test))

print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [23]:
image_folder4 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Orange"
csv_file4 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/orange_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Orange"


columns = csv_file4.columns
print(list(columns))

for i in range(0, len(csv_file4)):
    image_name = csv_file4.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file4.loc[i]["used_for"]
    
    image = cv2.imread(image_folder4 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([4])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([4])
    else:
        x.append(image)
        label.append([4])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [24]:
image_folder5 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Palm"
csv_file5 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/palm_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Palm"


columns = csv_file5.columns
print(list(columns))

for i in range(0, len(csv_file5)):
    image_name = csv_file5.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file5.loc[i]["used_for"]
    
    image = cv2.imread(image_folder5 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([5])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([5])
    else:
        x.append(image)
        label.append([5])

test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [25]:
image_folder6 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Persimmon"
csv_file6 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/persimmon_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Persimmon"


columns = csv_file6.columns
print(list(columns))

for i in range(0, len(csv_file6)):
    image_name = csv_file6.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file6.loc[i]["used_for"]
    
    image = cv2.imread(image_folder6 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([6])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([6])
    else:
        x.append(image)
        label.append([6])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [26]:
image_folder7 = "/kaggle/input/datasets/mahmoudshaheen1134/plant-leaf-image-dataset/New_data/Tomato"
csv_file7 = pd.read_csv("/kaggle/input/datasets/ragibmehedi/egypli-split/EGYPLI_split/tomato_split.csv")
json_folder = "/kaggle/input/datasets/ragibmehedi/egypli-annotated-json-files/annotated/Tomato"


columns = csv_file7.columns
print(list(columns))

for i in range(0, len(csv_file7)):
    image_name = csv_file7.loc[i]["image"]
    json_name = image_name.replace(".jpg", ".json")
    fold = csv_file7.loc[i]["used_for"]
    
    image = cv2.imread(image_folder7 +'/'+ image_name)
    image_original = image
    #image = cv2.resize(image, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    image = cv2.resize(image, (img_size, img_size))
    image = image.astype("float32") / 255.0

    json_file = json_folder + '/' +json_name

    if fold == "testing":
        x_test.append(image)
        label_test.append([7])
        mask = create_mask_from_json( json_file , image_original.shape[0])
        #mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        mask = cv2.resize(mask, (img_size, img_size))
        mask_test.append(mask)
    elif fold == "validation":
        x_validation.append(image)
        label_validation.append([7])
    else:
        x.append(image)
        label.append([7])


test_range.append(len(x_test))
print("Hello world.")

['Unnamed: 0', 'image', 'used_for']
Hello world.


In [27]:
x = np.asarray(x)
x_validation = np.asarray(x_validation)
x_test = np.asarray(x_test)

print(len(x)," ", len(x_validation)," ", len(x_test))

2868   359   361


In [28]:
from tensorflow.keras.utils import to_categorical 

label = np.array(label)
label = to_categorical(label , 8 )
print(len(label))



label_validation = np.array(label_validation)
label_validation = to_categorical(label_validation , 8 )
print(len(label_validation))

label_test = np.array(label_test)
label_test = to_categorical(label_test , 8 )
print(len(label_test))

2868
359
361


In [29]:
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Define input layer (Xception standard input size)
inputs = Input(shape=(299, 299, 3))

# Load Xception base model without the top layer
base_model = Xception(weights="imagenet", include_top=False, input_tensor=inputs)
base_model.trainable = False  # Freeze base model

# Add custom layers on top
y = base_model.output
y = GlobalAveragePooling2D()(y)
y = Dropout(0.3)(y)
y = Dense(128, activation="relu")(y)
outputs = Dense(8, activation="softmax")(y)

# Create the functional model
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=5e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 149, 149,  │        864 │ input_layer_1[0]… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 149, 149,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 149, 149,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 147, 147,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 147, 147,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 147, 147,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 147, 147,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 147, 147,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 147, 147,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 147, 147,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 147, 147,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 74, 74,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 74, 74,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 74, 74,    │        512 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 74, 74,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 74, 74,    │          0 │ add_12[0][0]    

 Total params: 21,124,784 (80.58 MB)

 Trainable params: 263,304 (1.00 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [30]:
checkpoint1 = tf.keras.callbacks.ModelCheckpoint(
    "Xception_EGYPLI.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

history1 = model.fit(
    x, label,
    validation_data=(x_validation, label_validation),
    epochs=100,
    batch_size=16,
    callbacks=[checkpoint1]
)

Epoch 1/100
180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.3605 - loss: 1.8671
Epoch 1: val_accuracy improved from -inf to 0.77437, saving model to Xception_EGYPLI.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 73s 319ms/step - accuracy: 0.3615 - loss: 1.8657 - val_accuracy: 0.7744 - val_loss: 1.1462
Epoch 2/100
179/180 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.7789 - loss: 1.0537
Epoch 2: val_accuracy improved from 0.77437 to 0.90808, saving model to Xception_EGYPLI.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 24s 131ms/step - accuracy: 0.7794 - loss: 1.0521 - val_accuracy: 0.9081 - val_loss: 0.6455
Epoch 3/100
179/180 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - accuracy: 0.8868 - loss: 0.6592
Epoch 3: val_accuracy improved from 0.90808 to 0.94150, saving model to Xception_EGYPLI.keras
180/180 ━━━━━━━━━━━━━━━━━━━━ 23s 130ms/step - accuracy: 0.8871 - loss: 0.6581 - val_accuracy: 0.9415 - val_loss: 0.4187
Epoch 4/100
179/180 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.9450 - loss: 0.4079
Epoch

In [31]:
test_loss, test_acc = model.evaluate(x_test, label_test)
print("Test Accuracy:", test_acc)

2026-05-22 04:09:51.928590: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=2} for conv %cudnn-conv.78 = (f32[32,128,147,147]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,128,147,147]{3,2,1,0} %bitcast.4820, f32[128,1,3,3]{3,2,1,0} %bitcast.4824), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=128, custom_call_target="__cudnn$convForward", metadata={op_type="DepthwiseConv2dNative" op_name="functional_1_1/block2_sepconv2_1/separable_conv2d/depthwise" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-05-22 04:09:51.981992: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.053521653s
Trying 

11/12 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step - accuracy: 0.9997 - loss: 0.0025

2026-05-22 04:10:23.498992: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 04:10:23.722193: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 04:10:24.485277: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 04:10:24.672127: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 04:10:25.296768: E external/local_xla/xla/stream_

12/12 ━━━━━━━━━━━━━━━━━━━━ 54s 2s/step - accuracy: 0.9994 - loss: 0.0029 
Test Accuracy: 0.9972299337387085


In [32]:
from tensorflow.keras.models import load_model

model_path = "/kaggle/working/Xception_EGYPLI.keras"
model = load_model(model_path)
print("Model loaded successfully!")

Model loaded successfully!


In [33]:
test_loss, test_accuracy = model.evaluate(x_test, label_test )
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

12/12 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.9989 - loss: 0.0176 
Test Accuracy: 0.9945
Test Loss: 0.0244


In [34]:
predictions = model.predict(x_test)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(label_test, axis=1)

12/12 ━━━━━━━━━━━━━━━━━━━━ 25s 1s/step 


In [35]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes))

# Generate a confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(true_classes, predicted_classes))


Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        52
           1       1.00      1.00      1.00        34
           2       1.00      1.00      1.00        52
           3       1.00      1.00      1.00        52
           4       0.98      1.00      0.99        55
           5       1.00      1.00      1.00        47
           6       1.00      0.98      0.99        53
           7       1.00      0.94      0.97        16

    accuracy                           0.99       361
   macro avg       1.00      0.99      0.99       361
weighted avg       0.99      0.99      0.99       361


Confusion Matrix:
[[52  0  0  0  0  0  0  0]
 [ 0 34  0  0  0  0  0  0]
 [ 0  0 52  0  0  0  0  0]
 [ 0  0  0 52  0  0  0  0]
 [ 0  0  0  0 55  0  0  0]
 [ 0  0  0  0  0 47  0  0]
 [ 0  0  0  0  1  0 52  0]
 [ 1  0  0  0  0  0  0 15]]


In [36]:
# while(True):
#     a = 5